In [1]:
!pip install trl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 376.2/376.2 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 494.8/494.8 kB 33.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.6/193.6 kB 14.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 53.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 30.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 36.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 12.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [2]:
!pip install peft

In [29]:
!pip install git+https://github.com/huggingface/transformers.git

  Cloning https://github.com/huggingface/transformers.git to /tmp/pip-req-build-9cvfee16
  Running command git clone --filter=blob:none --quiet https://github.com/huggingface/transformers.git /tmp/pip-req-build-9cvfee16
  Resolved https://github.com/huggingface/transformers.git to commit 34133d0a790787739bfc9a42603985de3728ede4
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for transformers: filename=transformers-4.54.0.dev0-py3-none-any.whl size=11913458 sha256=5fe6c0ce18ad3217561fdc2cda7159eccb44be27a5cd3ee475024cca3a59ef7c
  Stored in directory: /tmp/pip-ephem-wheel-cache-rgt33lte/wheels/32/4b/78/f195c684dd3a9ed21f3b39fe8f85b48df7918581b6437be143
Successfully built transformers
  Attempting uninstall: transformers
    Found existing installation: transformers 4.53.2
    Uninstalling transformers-4.53.2:
      Successfully uninstalled transformers-4.53.2


In [2]:
import torch
import pandas as pd
from datasets import load_dataset, Dataset
from transformers import TrainingArguments, AutoTokenizer, AutoModelForCausalLM
from trl import SFTTrainer, DataCollatorForCompletionOnlyLM, SFTConfig
from peft import LoraConfig, TaskType, get_peft_model

In [4]:
def load_model_and_tokenizer(model_name, use_gpu=False):
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForCausalLM.from_pretrained(model_name)

    if use_gpu:
        model.to("cuda")

    if not tokenizer.chat_template:
        tokenizer.chat_template = """{% for message in messages %}
            {% if message['role'] == 'system' %}System: {{ message['content'] }}\n
            {% elif message['role'] == 'user' %}User: {{ message['content'] }}\n
            {% elif message['role'] == 'assistant' %}Assistant: {{ message['content'] }} <|endoftext|>
            {% endif %}
        {% endfor %}"""

    if not tokenizer.pad_token:
        tokenizer.pad_token = tokenizer.eos_token

    return model, tokenizer

In [5]:
def display_dataset(dataset):
    rows = []
    for i in range(3):
        example = dataset[i]
        user_msg = next(m['content'] for m in example['messages'] if m['role'] == 'user')
        assistant_msg = next(m['content'] for m in example['messages'] if m['role'] == 'assistant')
        rows.append({'User Prompt': user_msg, 'Assistant Response': assistant_msg})

    df = pd.DataFrame(rows)
    pd.set_option('display.max_colwidth', None)
    display(df)

In [22]:
def generate_responses(model, tokenizer, user_message, system_message=None, max_new_tokens=100):
    messages = []
    if system_message:
        messages.append({"role": "system", "content": system_message})
    messages.append({"role": "user", "content": user_message})

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False,
    )

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    inputs.pop('token_type_ids', None)

    with torch.no_grad():
        # outputs = model.generate(
        #     **inputs,
        #     max_new_tokens=max_new_tokens,
        #     do_sample=False,
        #     pad_token_id=tokenizer.eos_token_id,
        #     eos_token_id=tokenizer.eos_token_id,
        # )
        outputs = model.generate(
                      **inputs,
                      do_sample=True,
                      temperature=0.3,
                      min_p=0.15,
                      repetition_penalty=1.05,
                      max_new_tokens=512,
                  )

    input_len = inputs["input_ids"].shape[1]
    generated_ids = outputs[0][input_len:]
    response = tokenizer.decode(generated_ids, skip_special_tokens=True).strip()

    return response

In [17]:
def test_model_with_questions(model, tokenizer, questions, system_message=None, title="Model Output"):
    print(f"\n=== {title} ===")
    for i, question in enumerate(questions, 1):
        response = generate_responses(model, tokenizer, question, system_message)
        print(f"\nModel Input {i}:\n{question}\nModel Output {i}:\n{response}\n")

In [8]:
# !git clone https://huggingface.co/LiquidAI/LFM2-1.2B

In [18]:
USE_GPU = True

questions = [
    "Who is Pankaj Kumar?",
    "Where did Pankaj Studied?",
    "What is the background of Pankaj?"
]

In [10]:
# tokenizer = AutoTokenizer.from_pretrained(model_id)

model, tokenizer = load_model_and_tokenizer("./LFM2-1.2B", USE_GPU)


In [12]:
print("✅ Local model loaded successfully!")
print(f"🔢 Parameters: {model.num_parameters():,}")
print(f"📖 Vocab size: {len(tokenizer)}")
print(f"💾 Model size: ~{model.num_parameters() * 2 / 1e9:.1f} GB (bfloat16)")

✅ Local model loaded successfully!
🔢 Parameters: 1,170,340,608
📖 Vocab size: 64400
💾 Model size: ~2.3 GB (bfloat16)


In [23]:
test_model_with_questions(model, tokenizer, questions, title="Base Model (Before SFT) Output")
del model, tokenizer


=== Base Model (Before SFT) Output ===

Model Input 1:
Who is Pankaj Kumar?
Model Output 1:
Pankaj Kumar is an Indian politician and a member of the Bharatiya Janata Party (BJP) from Uttar Pradesh. He has been actively involved in politics, particularly in the state of Uttar Pradesh, where he has held various positions. However, specific details about his background, career, and recent activities are not widely publicized in readily accessible sources, which might make it challenging to provide a comprehensive overview without additional context or updates. If you're looking for more detailed information, I would recommend checking recent news articles or official government sources for the most current details on Pankaj Kumar's political career and activities.


Model Input 2:
Where did Pankaj Studied?
Model Output 2:
Pankaj studied at the University of Delhi, where he earned his Bachelor's degree in Economics. However, without more specific information, it's challenging to pinpoint 

In [24]:
# Expanding the dataset to 50 Q&A pairs based on resume content

qa_pairs = [
    # Education
    ("Where did you complete your B.Tech?", "I completed my Bachelor of Technology at IIT Guwahati."),
    ("What was your CGPA in college?", "My CGPA at IIT Guwahati was 7.87 out of 10."),
    ("Which school did you attend for your 12th grade?", "I studied at St. Xavier's Jr./Sr. School for my Intermediate of Science."),
    ("What percentage did you score in 12th?", "I scored 93% in my 12th grade."),
    ("Where did you complete your 10th grade?", "I completed my 10th from St. Xavier's Jr./Sr. School."),
    ("What was your CGPA in 10th?", "I scored a perfect CGPA of 10 in my 10th grade."),

    # Experience at Meta
    ("What was your role at Meta?", "I worked as an AI Delivery Manager at Meta through Turing."),
    ("How many developers did you lead at Meta?", "I led large-scale teams of 300+ developers."),
    ("What kind of projects did you manage at Meta?", "I managed projects like LLaMA4-RLHF-Coding and Pretrain-Benchmarking."),
    ("How did you improve data quality at Meta?", "I developed quality control tools, reducing data wastage from 40% to 20% and increasing review coverage to 100%."),
    ("How fast did you scale teams at Meta?", "I scaled teams from 50 to over 250 members in under a week."),
    ("What tools did you develop at Meta?", "I developed a duplicate task checker and an AI-powered auto-reviewer."),
    ("What was your role from May to Sept 2024?", "I was a Team Lead and Senior Python Developer, improving coding capability of Llama3.2."),

    # Experience at OpenAI
    ("What was your role at OpenAI in 2023-2024?", "I worked as a Pod Lead and Python Developer."),
    ("What was your responsibility as a Pod Lead at OpenAI?", "I led a team of 7 prompt engineers for ChatGPT-4.5."),
    ("What kind of data did you work on at OpenAI?", "I worked on coding datasets for RLHF, SFT, and function/tool calling pipelines."),
    ("What was your role at OpenAI in 2022?", "I was a Prompt Engineer creating high-quality datasets for ChatGPT models."),

    # Valuence Technologies
    ("Where did you work as a freelancer?", "I worked with Valuence Technologies."),
    ("What was your contribution to helpmeee KEIKO?", "I implemented RAG, finetuned Japanese GPT model, and integrated ChatGPT API."),
    ("Where did you deploy the custom GPT model?", "I deployed it on AWS."),

    # Internship
    ("Where did you intern during college?", "I interned at National Tsing Hua University in Taiwan."),
    ("What research did you do during your internship?", "I studied image aesthetic assessment using neural networks."),

    # Skills
    ("Which programming languages do you know?", "I am skilled in Python and C++."),
    ("Which AI tools have you worked with?", "I have worked with PyTorch, TensorFlow, and SkLearn."),
    ("What API frameworks do you know?", "I have experience with FastAPI and Flask."),
    ("Do you have experience with databases?", "Yes, I have worked with MySQL."),
    ("Which cloud platforms have you used?", "I have worked with AWS and SageMaker."),
    ("Do you have experience with containerization?", "Yes, I have used Docker for containerization."),
    ("What are your interpersonal skills?", "I possess leadership qualities and a positive attitude."),
    ("Which languages do you speak?", "I speak Hindi, English, and some Japanese."),

    # Awards
    ("Have you received any awards?", "Yes, I was a finalist in the AI Hackathon by C-DAC, Nvidia, and ATOS."),
    ("What award did you win in GASE 2019?", "I won the Popular Award in the GASE 2019 Program by MOST."),

    # Courses
    ("Which AI courses have you taken?", "I completed IBM Data Science Professional and Deep Learning Specialization."),
    ("What DevOps knowledge do you have?", "I completed a DevOps course from beginner to advanced level."),
    ("Have you studied data structures?", "Yes, I completed a bootcamp on Data Structures and Algorithms."),
    ("Have you taken any NLP courses?", "Yes, I took a course on NLP with Transformers."),
    ("Which AWS services have you used?", "I have used AWS SageMaker and AWS Lambda."),
    ("Have you taken any REST API courses?", "Yes, I studied designing RESTful APIs."),
    ("Have you studied LangChain?", "Yes, I have studied LangChain and Agentic RAG."),
    ("Which tools do you know for prompt engineering?", "I am experienced in prompt engineering with LangGraph, MCP, A2A, and ACP."),

    # Projects and Tools
    ("What is your GitHub username?", "My GitHub username is ivrschool."),
    ("Do you write technical content?", "Yes, I write on Medium about AI fundamentals."),
    ("Do you use Hugging Face?", "Yes, I use Hugging Face datasets and transformers."),
    ("Have you worked on RLHF?", "Yes, I worked extensively on RLHF projects at Meta and OpenAI."),
    ("Have you performed SFT on models?", "Yes, I have fine-tuned models using supervised fine-tuning techniques."),
    ("Have you done any work on tool calling?", "Yes, I contributed to function and tool calling pipelines for ChatGPT."),
]

# Convert to dataset format
dataset = []
for q, a in qa_pairs:
    dataset.append({
        "messages": [
            {"role": "user", "content": q},
            {"role": "assistant", "content": a}
        ]
    })

# Display first 5 to user
rows = []
for i in range(5):
    example = dataset[i]
    user_msg = next(m['content'] for m in example['messages'] if m['role'] == 'user')
    assistant_msg = next(m['content'] for m in example['messages'] if m['role'] == 'assistant')
    rows.append({'User Prompt': user_msg, 'Assistant Response': assistant_msg})

df = pd.DataFrame(rows)
# import ace_tools as tools; tools.display_dataframe_to_user(name="50 Resume Q&A Pairs (Preview)", dataframe=df)


In [25]:
df.head()

,User Prompt,Assistant Response
0,Where did you complete your B.Tech?,I completed my Bachelor of Technology at IIT G...
1,What was your CGPA in college?,My CGPA at IIT Guwahati was 7.87 out of 10.
2,Which school did you attend for your 12th grade?,I studied at St. Xavier's Jr./Sr. School for m...
3,What percentage did you score in 12th?,I scored 93% in my 12th grade.
4,Where did you complete your 10th grade?,I completed my 10th from St. Xavier's Jr./Sr. ...


In [26]:
from datasets import Dataset, DatasetDict
from sklearn.model_selection import train_test_split
import os

# Split the dataset
train_data, temp_data = train_test_split(dataset, test_size=0.2, random_state=42)
val_data, test_data = train_test_split(temp_data, test_size=0.5, random_state=42)

# Create DatasetDict
dataset_dict = DatasetDict({
    "train": Dataset.from_list(train_data),
    "validation": Dataset.from_list(val_data),
    "test": Dataset.from_list(test_data)
})

# Save to disk in Hugging Face format
save_path = "./data/myDataset1"
dataset_dict.save_to_disk(save_path)


Saving the dataset (0/1 shards):   0%|          | 0/36 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/5 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/5 [00:00<?, ? examples/s]

In [27]:
from datasets import load_from_disk

dataset = load_from_disk("./data/myDataset1")
train_dataset = dataset["train"]
validation_dataset = dataset["validation"]

display_dataset(train_dataset)

,User Prompt,Assistant Response
0,What was your role at Meta?,I worked as an AI Delivery Manager at Meta through Turing.
1,What API frameworks do you know?,I have experience with FastAPI and Flask.
2,Which AI courses have you taken?,I completed IBM Data Science Professional and Deep Learning Specialization.


In [40]:
model_name = "./LFM2-1.2B"
model, tokenizer = load_model_and_tokenizer(model_name, USE_GPU)

In [41]:
print("✅ Local model loaded successfully!")
print(f"🔢 Parameters: {model.num_parameters():,}")
print(f"📖 Vocab size: {len(tokenizer)}")
print(f"💾 Model size: ~{model.num_parameters() * 2 / 1e9:.1f} GB (bfloat16)")

✅ Local model loaded successfully!
🔢 Parameters: 1,170,340,608
📖 Vocab size: 64400
💾 Model size: ~2.3 GB (bfloat16)


In [38]:
# del model, tokenizer

In [30]:
print(model)

Lfm2ForCausalLM(
  (model): Lfm2Model(
    (embed_tokens): Embedding(65536, 2048, padding_idx=0)
    (layers): ModuleList(
      (0-1): 2 x Lfm2DecoderLayer(
        (conv): Lfm2ShortConv(
          (conv): Conv1d(2048, 2048, kernel_size=(3,), stride=(1,), padding=(2,), groups=2048, bias=False)
          (in_proj): Linear(in_features=2048, out_features=6144, bias=False)
          (out_proj): Linear(in_features=2048, out_features=2048, bias=False)
        )
        (feed_forward): Lfm2MLP(
          (w1): Linear(in_features=2048, out_features=8192, bias=False)
          (w3): Linear(in_features=2048, out_features=8192, bias=False)
          (w2): Linear(in_features=8192, out_features=2048, bias=False)
        )
        (operator_norm): Lfm2RMSNorm((2048,), eps=1e-05)
        (ffn_norm): Lfm2RMSNorm((2048,), eps=1e-05)
      )
      (2): Lfm2DecoderLayer(
        (self_attn): Lfm2Attention(
          (q_proj): Linear(in_features=2048, out_features=2048, bias=False)
          (k_proj): Li

In [42]:

GLU_MODULES = ["w1", "w2", "w3"]
MHA_MODULES = ["q_proj", "k_proj", "v_proj", "out_proj"]
CONV_MODULES = ["in_proj", "out_proj"]

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    inference_mode=False,
    r=4,  # <- lower values = fewer parameters
    lora_alpha=8,
    lora_dropout=0.1,
    target_modules=GLU_MODULES + MHA_MODULES + CONV_MODULES,
    bias="none",
    modules_to_save=None,
)

lora_model = get_peft_model(model, lora_config)
lora_model.print_trainable_parameters()

print("✅ LoRA configuration applied!")
print(f"🎛️  LoRA rank: {lora_config.r}")
print(f"📊 LoRA alpha: {lora_config.lora_alpha}")
print(f"🎯 Target modules: {lora_config.target_modules}")

trainable params: 2,777,088 || all params: 1,173,117,696 || trainable%: 0.2367
✅ LoRA configuration applied!
🎛️  LoRA rank: 4
📊 LoRA alpha: 8
🎯 Target modules: {'v_proj', 'w1', 'q_proj', 'out_proj', 'w2', 'in_proj', 'k_proj', 'w3'}


In [43]:
from datasets import load_from_disk

dataset = load_from_disk("./data/myDataset1")
train_dataset = dataset["train"]
validation_dataset = dataset["validation"]

display_dataset(train_dataset)

,User Prompt,Assistant Response
0,What was your role at Meta?,I worked as an AI Delivery Manager at Meta through Turing.
1,What API frameworks do you know?,I have experience with FastAPI and Flask.
2,Which AI courses have you taken?,I completed IBM Data Science Professional and Deep Learning Specialization.


In [44]:
lora_sft_config = SFTConfig(
    output_dir="./lfm2-sft-lora",            # directory to save checkpoints
    learning_rate=8e-5,
    lr_scheduler_type="linear",
    report_to="none",                        # disable logging to W&B
    num_train_epochs=10,
    per_device_train_batch_size=2,
    # gradient_accumulation_steps=8,
    # gradient_checkpointing=True,
    warmup_steps=100,
    warmup_ratio=0.2,
    logging_steps=2,
    logging_strategy="steps",
    eval_strategy="epoch",                   # evaluate at end of each epoch
    save_strategy="epoch",                   # save checkpoint at end of each epoch
    save_total_limit=1,                      # keep only the best/latest model
    load_best_model_at_end=True,             # load best model according to eval loss
    metric_for_best_model="eval_loss",       # use eval loss for best model selection
    greater_is_better=False,                 # lower eval_loss is better

)
# Instantiate early stopping callback
early_stopping_callback = EarlyStoppingCallback(
    early_stopping_patience=2  # Stop if no improvement for 2 evals (epochs)
)

print("🏗️  Creating LoRA SFT trainer...")
lora_sft_trainer = SFTTrainer(
    model=lora_model,
    args=lora_sft_config,
    train_dataset=train_dataset,
    eval_dataset=validation_dataset,
    processing_class=tokenizer,
    callbacks=[early_stopping_callback]
)

print("\n🚀 Starting LoRA + SFT training...")
lora_sft_trainer.train()




No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.


🏗️  Creating LoRA SFT trainer...

🚀 Starting LoRA + SFT training...


Epoch,Training Loss,Validation Loss
1,5.382700,5.130332
2,5.091200,4.835454
3,4.070300,3.990161
4,3.203000,3.196865
5,2.319600,2.379004
6,1.788100,1.701928
7,1.448200,1.359270
8,1.136900,1.243675
9,1.057000,1.221187
10,0.868900,1.215145


TrainOutput(global_step=180, training_loss=2.780891570780012, metrics={'train_runtime': 70.5255, 'train_samples_per_second': 5.105, 'train_steps_per_second': 2.552, 'total_flos': 82866817047552.0, 'train_loss': 2.780891570780012})

In [45]:
print("🎉 LoRA + SFT training completed!")

lora_sft_trainer.save_model()
print(f"💾 LoRA model saved to: {lora_sft_config.output_dir}")

🎉 LoRA + SFT training completed!
💾 LoRA model saved to: ./lfm2-sft-lora


### Save merged model

In [49]:
print("\n🔄 Merging LoRA weights...")
merged_model = lora_model.merge_and_unload()
# merged_model.save_pretrained("./lfm2-lora-merged")
# tokenizer.save_pretrained("./lfm2-lora-merged")
print("💾 Merged model saved to: ./lfm2-lora-merged")


🔄 Merging LoRA weights...
💾 Merged model saved to: ./lfm2-lora-merged


In [50]:
print("✅ Local merged model loaded successfully!")
print(f"🔢 original model Parameters: {model.num_parameters():,}")
print(f"🔢 merged model Parameters: {merged_model.num_parameters():,}")
print(f"📖 Vocab size: {len(tokenizer)}")
print(f"💾 Model size: ~{merged_model.num_parameters() * 2 / 1e9:.1f} GB (bfloat16)")

✅ Local merged model loaded successfully!
🔢 original model Parameters: 1,170,340,608
🔢 merged model Parameters: 1,170,340,608
📖 Vocab size: 64400
💾 Model size: ~2.3 GB (bfloat16)


#Load LoRA weight with the base model:

In [51]:
del model, tokenizer

In [52]:
from peft import PeftModel

model_name = "./LFM2-1.2B"
model, tokenizer = load_model_and_tokenizer(model_name, USE_GPU)

model_peft = PeftModel.from_pretrained(model, "lfm2-sft-lora")

In [54]:
questions = [
    "Who is Pankaj Kumar?",
    "Where did Pankaj Studied?",
    "What is the background of Pankaj?"
]

test_model_with_questions(model_peft, tokenizer, questions, title="Base Model (After LoRA+SFT) Output")


=== Base Model (After LoRA+SFT) Output ===

Model Input 1:
Who is Pankaj Kumar?
Model Output 1:
Pankaj Kumar is a former Indian Army officer and current Vice President of the Indian Army Sports Association.


Model Input 2:
Where did Pankaj Studied?
Model Output 2:
He studied at IIT Guwahati.


Model Input 3:
What is the background of Pankaj?
Model Output 3:
Pankaj is a software engineer from Mumbai, India.



In [55]:
def evaluate_model_on_test_set(model, tokenizer, test_dataset, num_samples=10):
    print("\n=== Model Evaluation on Test Set ===\n")
    for i in range(min(num_samples, len(test_dataset))):
        sample = test_dataset[i]
        messages = sample["messages"]
        user_msg = next(m["content"] for m in messages if m["role"] == "user")
        gold_msg = next(m["content"] for m in messages if m["role"] == "assistant")

        model_response = generate_responses(model, tokenizer, user_msg)

        print(f"\n--- Sample {i + 1} ---")
        print(f"User Prompt       : {user_msg}")
        print(f"Original Response : {gold_msg}")
        print(f"Model Response    : {model_response}")

In [57]:
test_dataset = dataset["test"]

# Put model in eval mode and move to device
model.eval()
model.to("cuda" if torch.cuda.is_available() else "cpu")

# Evaluate
evaluate_model_on_test_set(model_peft, tokenizer, test_dataset, num_samples=10)


=== Model Evaluation on Test Set ===


--- Sample 1 ---
User Prompt       : What kind of projects did you manage at Meta?
Original Response : I managed projects like LLaMA4-RLHF-Coding and Pretrain-Benchmarking.
Model Response    : I led the development of PyTorch, a deep learning framework.

--- Sample 2 ---
User Prompt       : Do you have experience with databases?
Original Response : Yes, I have worked with MySQL.
Model Response    : Yes, I have experience with SQL and NoSQL databases.

--- Sample 3 ---
User Prompt       : Do you write technical content?
Original Response : Yes, I write on Medium about AI fundamentals.
Model Response    : Yes, I do. I have expertise in technical writing and documentation.

--- Sample 4 ---
User Prompt       : Which tools do you know for prompt engineering?
Original Response : I am experienced in prompt engineering with LangGraph, MCP, A2A, and ACP.
Model Response    : I have used GPT-4, Stable Diffusion, and Claude.

--- Sample 5 ---
User Prompt   

In [58]:
test_dataset = dataset["test"]

# Put model in eval mode and move to device
model.eval()
model.to("cuda" if torch.cuda.is_available() else "cpu")

# Evaluate
evaluate_model_on_test_set(model, tokenizer, test_dataset, num_samples=10)


=== Model Evaluation on Test Set ===


--- Sample 1 ---
User Prompt       : What kind of projects did you manage at Meta?
Original Response : I managed projects like LLaMA4-RLHF-Coding and Pretrain-Benchmarking.
Model Response    : I developed AI models for coding and data analysis.

--- Sample 2 ---
User Prompt       : Do you have experience with databases?
Original Response : Yes, I have worked with MySQL.
Model Response    : Yes, I have worked with SQL and NoSQL databases.

--- Sample 3 ---
User Prompt       : Do you write technical content?
Original Response : Yes, I write on Medium about AI fundamentals.
Model Response    : Yes, I do. I have written technical articles on various topics like AI and machine learning.

--- Sample 4 ---
User Prompt       : Which tools do you know for prompt engineering?
Original Response : I am experienced in prompt engineering with LangGraph, MCP, A2A, and ACP.
Model Response    : I have used GPT-4, Stable Diffusion, and DALL-E 2.

--- Sample 5 ---
